In [ ]:
# Enhanced SwapDeep Video with Mouth Mask
!pip install -U insightface onnxruntime-gpu opencv-python requests tqdm torch gradio --quiet

import os
import requests
import cv2
import numpy as np
import gradio as gr
from insightface.app import FaceAnalysis
from insightface.model_zoo import get_model
from tqdm import tqdm

def download(url, path):
    if not os.path.isfile(path):
        print(f"Downloading {os.path.basename(path)}...")
        r = requests.get(url, stream=True)
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk: f.write(chunk)
        print("Done.")

# Download model
download("https://huggingface.co/countfloyd/deepfake/resolve/main/inswapper_128.onnx", "inswapper_128.onnx")

# Initialize models
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
face_analyser = FaceAnalysis(providers=providers)
face_analyser.prepare(ctx_id=0, det_size=(640, 640))
swapper = get_model("inswapper_128.onnx", providers=providers, download=False)

print("✅ Models loaded for video processing")

In [ ]:
def create_mouth_mask(face, frame):
    """Create enhanced mouth mask for video"""
    mask = np.zeros(frame.shape[:2], dtype=np.uint8)
    if not hasattr(face, 'landmark_2d_106') or face.landmark_2d_106 is None:
        return mask, None, (0,0,0,0)
    
    landmarks = face.landmark_2d_106
    if len(landmarks) < 106:
        return mask, None, (0,0,0,0)
    
    # Extended mouth region
    mouth_indices = [65,66,62,70,69,18,19,20,21,22,23,24,0,8,7,6,5,4,3,2]
    mouth_landmarks = landmarks[mouth_indices].astype(np.int32)
    
    # Create adaptive mask
    center = np.mean(mouth_landmarks, axis=0)
    mouth_height = np.max(mouth_landmarks[:, 1]) - np.min(mouth_landmarks[:, 1])
    mouth_width = np.max(mouth_landmarks[:, 0]) - np.min(mouth_landmarks[:, 0])
    
    # Expand for movements
    expansion = 1.2 + (mouth_height / max(mouth_width, 1) * 0.3)
    expanded_landmarks = (mouth_landmarks - center) * expansion + center
    expanded_landmarks = np.clip(expanded_landmarks.astype(np.int32), 0, [frame.shape[1]-1, frame.shape[0]-1])
    
    # Create mask
    hull = cv2.convexHull(expanded_landmarks)
    cv2.fillConvexPoly(mask, hull, 255)
    mask = cv2.GaussianBlur(mask, (21, 21), 7)
    
    # Bounding box
    x, y, w, h = cv2.boundingRect(expanded_landmarks)
    padding = 10
    x, y = max(0, x-padding), max(0, y-padding)
    w, h = min(frame.shape[1]-x, w+2*padding), min(frame.shape[0]-y, h+2*padding)
    
    mouth_cutout = frame[y:y+h, x:x+w].copy() if w > 0 and h > 0 else None
    return mask, mouth_cutout, (x, y, x+w, y+h)

def process_video_with_mouth_mask(source_img_path, video_path, use_mouth_mask=True):
    """Process video with enhanced mouth mask"""
    if not os.path.exists(source_img_path) or not os.path.exists(video_path):
        return None, "❌ Files not found"
    
    # Load source face
    src_img = cv2.imread(source_img_path)
    faces_src = face_analyser.get(src_img)
    if not faces_src:
        return None, "❌ No face in source image"
    
    # Video setup
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    output_path = "enhanced_swapped_video.mp4"
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # Process frames
    for _ in tqdm(range(frame_count), desc="Processing video with mouth mask"):
        ret, frame = cap.read()
        if not ret:
            break
        
        faces_dst = face_analyser.get(frame)
        if faces_dst:
            # Face swap
            swapped = swapper.get(frame, faces_dst[0], faces_src[0], paste_back=True)
            
            # Apply mouth mask
            if use_mouth_mask:
                mask, mouth_cutout, box = create_mouth_mask(faces_dst[0], frame)
                if mouth_cutout is not None and box != (0,0,0,0):
                    x1, y1, x2, y2 = box
                    roi = swapped[y1:y2, x1:x2]
                    
                    if roi.shape[:2] == mouth_cutout.shape[:2]:
                        mask_roi = mask[y1:y2, x1:x2] / 255.0
                        blended = mouth_cutout * mask_roi[:,:,None] + roi * (1 - mask_roi[:,:,None])
                        swapped[y1:y2, x1:x2] = blended.astype(np.uint8)
            
            out.write(swapped)
        else:
            out.write(frame)
    
    cap.release()
    out.release()
    
    return output_path, f"✅ Video processed! {frame_count} frames with mouth mask"

print("✅ Video processing functions ready")

In [ ]:
# Launch Gradio interface for video
with gr.Blocks(title="Enhanced SwapDeep Video") as demo:
    gr.Markdown("# 🎬 Enhanced SwapDeep Video with Mouth Mask")
    gr.Markdown("Perfect for video face swapping with natural mouth preservation")
    
    with gr.Row():
        with gr.Column():
            source_img = gr.File(label="📷 Source Face Image", file_types=[".jpg", ".png", ".jpeg"])
            target_video = gr.File(label="🎥 Target Video", file_types=[".mp4", ".avi", ".mov"])
            mouth_mask_check = gr.Checkbox(label="🦷 Enable Mouth Mask", value=True)
            process_button = gr.Button("🔄 Process Video", variant="primary")
        
        with gr.Column():
            output_video = gr.File(label="✨ Enhanced Video Output")
            status_text = gr.Textbox(label="📊 Processing Status")
    
    gr.Markdown("""
    ### 📋 Video Processing Instructions:
    1. Upload source face image (clear, front-facing)
    2. Upload target video file
    3. Enable mouth mask for natural results
    4. Click Process Video (may take several minutes)
    
    **Enhanced Features:**
    - T4 GPU acceleration for faster processing
    - Perfect mouth mask for eating/drinking scenes
    - Temporal stability across frames
    - Preserves original video quality
    """)
    
    def process_uploaded_video(source_file, video_file, use_mouth_mask):
        if source_file is None or video_file is None:
            return None, "❌ Please upload both files"
        
        # Save uploaded files
        source_path = source_file.name
        video_path = video_file.name
        
        # Process video
        output_path, status = process_video_with_mouth_mask(source_path, video_path, use_mouth_mask)
        
        return output_path, status
    
    process_button.click(
        process_uploaded_video,
        inputs=[source_img, target_video, mouth_mask_check],
        outputs=[output_video, status_text]
    )

demo.launch(share=True, debug=True)

In [ ]:
# Alternative: Direct file path processing (for manual use)
def process_video_direct(source_img_path, video_path, output_name="enhanced_output.mp4"):
    """Direct video processing function"""
    
    # Load source
    src_img = cv2.imread(source_img_path)
    faces_src = face_analyser.get(src_img)
    assert faces_src, "No face in source!"
    
    # Video setup
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_name, fourcc, fps, (width, height))
    
    # Process with mouth mask
    for _ in tqdm(range(frame_count), desc="Enhanced face swapping"):
        ret, frame = cap.read()
        if not ret: break
        
        faces_dst = face_analyser.get(frame)
        if faces_dst:
            # Swap
            swapped = swapper.get(frame, faces_dst[0], faces_src[0], paste_back=True)
            
            # Mouth mask
            mask, mouth_cutout, box = create_mouth_mask(faces_dst[0], frame)
            if mouth_cutout is not None and box != (0,0,0,0):
                x1, y1, x2, y2 = box
                roi = swapped[y1:y2, x1:x2]
                if roi.shape[:2] == mouth_cutout.shape[:2]:
                    mask_roi = mask[y1:y2, x1:x2] / 255.0
                    blended = mouth_cutout * mask_roi[:,:,None] + roi * (1 - mask_roi[:,:,None])
                    swapped[y1:y2, x1:x2] = blended.astype(np.uint8)
            
            out.write(swapped)
        else:
            out.write(frame)
    
    cap.release()
    out.release()
    print(f"✅ Enhanced video saved: {output_name}")

# Example usage:
# process_video_direct("/content/source.jpg", "/content/target.mp4", "result.mp4")

print("✅ Direct processing function ready")
print("Use: process_video_direct('source.jpg', 'video.mp4', 'output.mp4')")